# 03 — Multi-Panel Pilatus3 2M-CdTe with the Σ=0 Gauge

Pilatus-style hybrid pixel detectors are tiled — the Pilatus3 2M-CdTe
in this work has **48 modules** in a 6×8 grid.  Each module has a
small position offset (~50 µm) and rotation (~1 mrad) from the
nominal grid that calibration must determine.

The fundamental challenge is **gauge invariance**: a uniform shift
of every panel by the same `(δy, δz)` is observationally identical
to a global beam-center shift.  Without breaking this gauge
explicitly, the per-panel covariances live at the regularisation
ridge floor and report no useful uncertainty.

This notebook walks through the Wright-2022 Σ=0 symmetric gauge
that the framework adopts, and shows the 10-orders-of-magnitude σ
collapse it produces.

> **Just want a calibration?** The one-shot `calibrate()` entry point
> (NB 00) now handles tiled detectors directly — build a layout and pass
> it:
> ```python
> from midas_calibrate_v2.forward.panels import PanelLayout
> layout = PanelLayout.regular(n_y=6, n_z=8, sy=243, sz=195,
>                              gap_y=(1, 7, 1, 7, 1), gap_z=17)
> result = calibrate(img, wavelength=0.172979, pxY=172.0, dark=dark,
>                    im_trans=(2,), panel_layout=layout,
>                    panel_mode='radius')   # or 'shift'
> ```
> `calibrate()` then runs two phases: monolithic geometry first, then a
> per-panel refinement with the global geometry frozen. `panel_mode='radius'`
> (default) refines a per-(panel,ring) δR — because the calibrant cost is
> purely radial, this nulls the per-module radial systematic directly and
> reaches the sub-100 µε (median/trimmed) regime. `panel_mode='shift'`
> refines the rigid (δy, δz, δθ) transform instead.
>
> **This notebook keeps the low-level spec path** because it also computes
> per-panel Bayesian σ (the Σ=0 gauge below), which the one-shot path does
> not expose.


In [ ]:
import os, sys, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pathlib import Path
import numpy as np
import torch
from PIL import Image

from midas_calibrate.params import CalibrationParams as V1Params
from midas_calibrate_v2.compat.from_v1 import (
    spec_from_v1_file, add_panel_parameters, add_panel_zero_sum_constraint,
)
from midas_calibrate_v2.parameters.transforms import Logit
from midas_calibrate_v2.pipelines.single_pv import autocalibrate_pv
from midas_calibrate_v2.loss.pseudo_strain import pseudo_strain_residual
from midas_calibrate_v2.loss.constraints import zero_sum_residual
from midas_calibrate_v2.loss.robust_trim import multfactor_trim
from midas_calibrate_v2.inference.laplace import fisher_at_map

BASE = Path(os.environ.get('V2_TEST_BASE', '/tmp/midas_v2_test'))
sys.path.insert(0, str(BASE))
from run_v2_full import (
    detect_pilatus_panels, _load_panel_shifts, _apply_panel_shifts_to_spec,
    _load_image,
)

ps_path = BASE / 'CeO2_Pil_100x100_att000_650mm_71p676keV_001956.tifps.txt'
img_path = BASE / 'CeO2_Pil_100x100_att000_650mm_71p676keV_001956.tif'
dark_path = BASE / 'dark_CeO2_Pil_100x100_att000_650mm_71p676keV_001975.tif'
panel_path = BASE / 'CeO2_Pil_100x100_att000_650mm_71p676keV_panel_shifts.txt'

assert ps_path.exists(), f'Pilatus dataset not found at {ps_path}'
print('Pilatus dataset found.')


## Detect the panel layout from the raw image

`detect_pilatus_panels` infers the gap structure from the dark
columns/rows in the raw image — useful when you don't have an
explicit module-map file.


In [ ]:
v1 = V1Params.from_file(ps_path)
if v1.RBinSize <= 0: v1.RBinSize = 0.25
if v1.EtaBinSize <= 0: v1.EtaBinSize = 5.0
v1.MaxRingRad = max(v1.MaxRingRad, v1.RhoD / max(v1.pxY, 1.0))
v1.Width = max(v1.Width, 800.0)

img = _load_image(img_path, im_trans=[2])
dark = np.array(Image.open(str(dark_path))).astype(np.float64)
raw = np.array(Image.open(str(img_path)))[::-1, :].copy()
layout = detect_pilatus_panels(raw, gap_thresh=0)
print(f'Layout: {layout.n_panels_y} × {layout.n_panels_z} panels = '
      f'{layout.n_panels()} modules total')
print(f'Module size: {layout.panel_size_y} × {layout.panel_size_z} px')
print(f'Inter-module gaps: y={layout.gaps_y}, z={layout.gaps_z}')


## Build the spec with per-panel parameters and the Σ=0 gauge

`add_panel_parameters` injects four per-panel blocks:
- `panel_delta_yz` — [N, 2] in-plane shift in pixels
- `panel_delta_theta` — [N] in-plane rotation in degrees
- `panel_delta_lsd` — [N] out-of-plane offset in µm
- `panel_delta_p2` — [N] per-panel additive distortion correction

`add_panel_zero_sum_constraint` enables the symmetric Σ=0 gauge:
the residual closure appends `√λ · Σ panel_delta_*` rows so every
per-panel block has zero sum.  At λ=10⁶ this is effectively a hard
constraint; at finite λ it acts as a Bayesian prior on the gauge
direction.


In [ ]:
spec = spec_from_v1_file(ps_path)
add_panel_parameters(
    spec, n_panels=layout.n_panels(),
    tol_shift_px=1.0, tol_rot_deg=1.0,
    tol_lsd_um=500.0, tol_p2=5e-3,
    enable_lsd=True, enable_p2=True,
)
shifts = _load_panel_shifts(panel_path, n_panels=layout.n_panels())
if shifts is not None:
    _apply_panel_shifts_to_spec(spec, shifts)
for nm in ('BC_y', 'BC_z'):
    p = spec.parameters[nm]; cur = float(p.init)
    p.bounds = (cur - 60.0, cur + 60.0); p.transform = Logit(*p.bounds)

# Σ=0 gauge — symmetric, no special reference panel
add_panel_zero_sum_constraint(spec, lambda_zs=1e6)

n_refined = sum(p.numel for p in spec.parameters.values() if p.refined)
print(f'Spec: {n_refined} refined parameters '
      f'({layout.n_panels()} modules × 5 DOF + headline geometry)')


In [ ]:
print('Running production pipeline (Pilatus, ~30 s)…')
t0 = time.time()
res = autocalibrate_pv(
    v1, img, dark=dark, spec=spec, panel_layout=layout,
    n_iter=1, half_window_px=4.0, snr_min=8.0,
    trim_mode='stratified_multfactor', trim_residual_pct=2.0,
    reuse_fits=True, lm_max_iter=1, verbose=False,
)
print(f'pipeline: {time.time()-t0:.0f} s')
fits = res.fits_final
truth_unp = res.unpacked

# Trim to define the σ_r-determining set
with torch.no_grad():
    r0 = pseudo_strain_residual(
        fits.Y_pix, fits.Z_pix, fits.ring_two_theta_deg, truth_unp,
        rho_d=fits.rho_d, panel_layout=layout, panel_idx=fits.panel_idx,
    )
    keep, _ = multfactor_trim(r0, factor=2.0)
sigma_r = float(((r0 * r0).mean()) ** 0.5)
print(f'σ_r = {sigma_r:.3e}, kept {int(keep.sum())}/{len(r0)}')


## Compute Laplace σ — with vs without the Σ=0 gauge

The same `fisher_at_map` call, two times:
1. **Fix-one-panel gauge** (`spec.fix_panel_id = 0`, no zero-sum) —
   the legacy v1-style choice.  Per-panel σ saturates at the ridge
   floor.
2. **Σ=0 gauge** — what we just enabled.  Per-panel σ collapses by
   ~10 orders of magnitude to the data-determined values.


In [ ]:
def res_fn_data_only(unp, _Y=fits.Y_pix[keep], _Z=fits.Z_pix[keep],
                     _tt=fits.ring_two_theta_deg[keep],
                     _pid=fits.panel_idx[keep]):
    return pseudo_strain_residual(
        _Y, _Z, _tt, unp, rho_d=fits.rho_d,
        panel_layout=layout, panel_idx=_pid,
    )

# (A) Fix-one-panel gauge — for comparison
spec_A = spec_from_v1_file(ps_path)
add_panel_parameters(spec_A, n_panels=layout.n_panels(),
                     tol_shift_px=4.0, tol_rot_deg=2.0,
                     tol_lsd_um=2000.0, tol_p2=2e-2,
                     enable_lsd=True, enable_p2=True)
if shifts is not None: _apply_panel_shifts_to_spec(spec_A, shifts)
for nm in ('BC_y', 'BC_z'):
    p = spec_A.parameters[nm]; cur = float(p.init)
    p.bounds = (cur - 200.0, cur + 200.0); p.transform = Logit(*p.bounds)
spec_A.fix_panel_id = 0
lap_A = fisher_at_map(spec_A, res_fn_data_only, truth_unp,
                      sigma_r=sigma_r, ridge=1e-6,
                      dtype=torch.float64, device='cpu')

# (B) Σ=0 gauge — re-use closure but add zero-sum residual rows
def res_fn_zero_sum(unp):
    r = res_fn_data_only(unp)
    zs = zero_sum_residual(unp, lambda_zs=1e6)
    return torch.cat([r, zs]) if zs.numel() > 0 else r

spec_B = spec   # already configured with zero-sum
lap_B = fisher_at_map(spec_B, res_fn_zero_sum, truth_unp,
                      sigma_r=sigma_r, ridge=1e-6,
                      dtype=torch.float64, device='cpu')
print('Laplace done (A: fix-panel, B: Σ=0)')


## The σ collapse

The numbers below are **median per-panel σ** in logit-domain units.
Only the **ratio** A/B is interpretable as the gauge-null collapse
signature; the absolute scale shifts with the bound width.


In [ ]:
def _flat(lap):
    out = []
    for n, o, s in zip(lap.refined_names, lap.refined_offsets, lap.refined_sizes):
        for k in range(s):
            out.append(f'{n}[{k}]' if s > 1 else n)
    return out

def block_med(lap, blk):
    flat = _flat(lap)
    sigma = lap.sigma_per_dim.detach().cpu().numpy()
    sigs = np.array([s for nm, s in zip(flat, sigma)
                      if nm.startswith(blk + '[') or nm == blk])
    return float(np.median(sigs)) if len(sigs) else float('nan')

print(f'{"per-panel block":<22s}  {"A: fix-panel":>14s}  '
      f'{"B: Σ=0":>14s}  {"ratio A/B":>14s}')
for blk in ('panel_delta_yz', 'panel_delta_theta',
            'panel_delta_lsd', 'panel_delta_p2'):
    a = block_med(lap_A, blk)
    b = block_med(lap_B, blk)
    print(f'  {blk:<22s}  {a:>14.3e}  {b:>14.3e}  {a/b:>14.3e}')


The Σ=0 gauge collapses per-panel σ by **~10 orders of magnitude**
on every per-panel block.  This is the difference between "cannot
be reported" and "data-determined".

## Soft prior on the gauge — sensitivity sweep

The default `lambda_zs=1e6` makes the Σ=0 constraint effectively
hard.  At softer λ the gauge becomes a Bayesian prior of stddev
`1/√λ`.  Per-panel σ scales as `λ^(-1/2)` — the signature of a
prior-dominated direction.  See `tab:sigma_collapse` in the paper
for the full λ ∈ {10⁶, 10², 1, 10⁻²} sweep.

## What if the manufacturer prior is informative?

`add_panel_parameters` only sets bounds, not Gaussian priors.  To
add a hierarchical prior matching the manufacturer's panel-installation
tolerance:

```python
from midas_calibrate_v2.parameters.parameter import GaussianPrior
spec.parameters['panel_delta_yz'].prior = GaussianPrior(0.0, 0.3)  # ±0.3 px
spec.parameters['panel_delta_theta'].prior = GaussianPrior(0.0, 0.05)  # ±0.05°
spec.parameters['panel_delta_lsd'].prior = GaussianPrior(0.0, 100.0)  # ±100 µm
spec.parameters['panel_delta_p2'].prior = GaussianPrior(0.0, 1e-4)
```

On full-arc Pilatus the prior never binds (the data dominates by
several orders of magnitude); see notebook **04** for the case
where priors actually do bind, and `dev/paper/runners/run_hierarchical_priors_sparse.py`
for the sparse-arc binding regime.
